# Part 2：迴歸 — 參考解答

使用專案 `.venv` Kernel。六個任務依序完成，前面的變數會供後面使用。所有資料為合成教學資料。


In [1]:
# 作業自帶環境與資料載入，不需要先執行主教材。
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

cwd = Path.cwd().resolve()
lesson_dir = next((p / "regression" for p in [cwd, *cwd.parents]
                   if (p / "regression/data/rental_housing.csv").exists()), None)
if lesson_dir is None:
    raise FileNotFoundError("找不到作業 CSV，請開啟專案資料夾。")
df = pd.read_csv(lesson_dir / "data/rental_housing.csv")
display(df.head())


,property_id,area_ping,building_age,distance_mrt_m,bedrooms,floor,district,has_elevator,monthly_rent
0,H0001,15.4,32.0,2196.0,1.0,15.0,West,Yes,16414
1,H0002,10.1,5.0,553.0,1.0,14.0,Central,Yes,19425
2,H0003,51.8,24.0,412.0,4.0,2.0,East,Yes,50537
3,H0004,35.0,7.0,1842.0,4.0,7.0,East,Yes,35862
4,H0005,46.8,45.0,520.0,4.0,8.0,West,Yes,40165


## 作業 1｜資料準備與切分

明確指定 Features，排除 property_id、monthly_rent。先切 20% 測試，再從剩下資料切 25% 作驗證，兩次 random_state=42。
印出三組筆數，檢查無索引交集。請說明為什麼不對連續租金直接 stratify=y。
**驗收：** 360 / 120 / 120 筆，X 與 y 索引對齊，尚未補值。


In [2]:
# 指定欄位，不把識別碼和答案當輸入。
numeric_features = ["area_ping", "building_age", "distance_mrt_m", "bedrooms", "floor"]
categorical_features = ["district", "has_elevator"]
feature_columns = numeric_features + categorical_features
X, y = df[feature_columns].copy(), df["monthly_rent"].copy()
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42)
print(len(X_train), len(X_valid), len(X_test))
assert X_train.index.equals(y_train.index)
assert set(X_train.index).isdisjoint(X_valid.index)
assert set(X_dev.index).isdisjoint(X_test.index)
print("連續租金通常每個值樣本少，不是可直接用於分層的分類標籤。")


360 120 120
連續租金通常每個值樣本少，不是可直接用於分層的分類標籤。


## 作業 2｜四個指標與業務說明

實際值 [15000, 25000, 35000, 45000]，預測 [16000, 23000, 39000, 40000]。
計算 MAE、MSE、RMSE、R²，並寫一句給業務看的 MAE 說明。
**驗收：** MAE=3000 元、MSE=11500000 元平方、RMSE 約3391.165 元、R²=0.908。不得稱 R² 為準確率。


In [3]:
# 保持每筆答案與預測順序一致。
actual = np.array([15000, 25000, 35000, 45000])
predicted = np.array([16000, 23000, 39000, 40000])
metrics = {
    "MAE": mean_absolute_error(actual, predicted),
    "MSE": mean_squared_error(actual, predicted),
    "RMSE": root_mean_squared_error(actual, predicted),
    "R2": r2_score(actual, predicted),
}
display(pd.Series(metrics))
assert np.isclose(metrics["MAE"], 3000)
assert np.isclose(metrics["R2"], 0.908)
print("這四筆資料的預測月租金平均差 3000 元；不是每筆誤差都小於 3000 元。")


MAE     3.000000e+03
MSE     1.150000e+07
RMSE    3.391165e+03
R2      9.080000e-01
dtype: float64

這四筆資料的預測月租金平均差 3000 元；不是每筆誤差都小於 3000 元。


## 作業 3｜簡單線性與多元線性

只用面積建立簡單線性 Pipeline，再用全部 Features 建立多元線性 Pipeline。
數值 median 補值，類別 most_frequent 補值 + One-Hot。兩者只 fit 訓練集，比較驗證 MAE。
**驗收：** 輸出兩個 MAE；每個模型預測120筆且皆有限值；可解釋增加資訊不保證一定改善。


In [4]:
# 每次呼叫建立新前處理，避免共用已訓練物件。
def make_preprocessor(scale=False):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ])
simple = Pipeline([
    ("prep", ColumnTransformer([("area", SimpleImputer(strategy="median"), ["area_ping"])])),
    ("reg", LinearRegression()),
])
multiple = Pipeline([("prep", make_preprocessor()), ("reg", LinearRegression())])
for name, estimator in [("simple", simple), ("multiple", multiple)]:
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_valid)
    assert len(pred) == 120 and np.isfinite(pred).all()
    print(name, "validation MAE:", mean_absolute_error(y_valid, pred))


simple validation MAE: 3503.7191596094044
multiple validation MAE: 2013.2831067543777


## 作業 4｜多項式與 SVR

讀取 advertising_curve.csv，以 random_state=42 切25%驗證。比較 degree=1、2、6 的驗證 MAE，列出展開欄數。
另用房屋訓練集建立縮放 X/y 的 SVR（C=10、epsilon=0.1），預測房屋驗證集。
**驗收：** 多項式只有一個數值輸入，欄數為1/2/6；SVR 輸出還原為元；說明 epsilon 是縮放單位。


In [5]:
# 多項式為獨立曲線案例，避免與房屋變數混淆。
curve = pd.read_csv(lesson_dir / "data/advertising_curve.csv")
cx_train, cx_valid, cy_train, cy_valid = train_test_split(
    curve[["ad_budget_thousand"]], curve["inquiries"], test_size=0.25, random_state=42
)
rows = []
for degree in [1, 2, 6]:
    estimator = Pipeline([
        ("scale", StandardScaler()),
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("reg", LinearRegression()),
    ]).fit(cx_train, cy_train)
    rows.append({"degree": degree, "columns": estimator.named_steps["poly"].n_output_features_,
                 "MAE": mean_absolute_error(cy_valid, estimator.predict(cx_valid))})
display(pd.DataFrame(rows))
# y 標準化由包裝器處理，predict 自動還原。
svr = TransformedTargetRegressor(
    regressor=Pipeline([("prep", make_preprocessor(scale=True)), ("reg", SVR(C=10, epsilon=0.1))]),
    transformer=StandardScaler(),
).fit(X_train, y_train)
print("SVR MAE (元):", mean_absolute_error(y_valid, svr.predict(X_valid)))
print("epsilon=0.1 是標準化後的 y 單位。")


,degree,columns,MAE
0,1,1,9.586251
1,2,2,5.170050
2,6,6,5.028008


SVR MAE (元): 2354.3474497363272
epsilon=0.1 是標準化後的 y 單位。


## 作業 5｜樹與森林的訓練/驗證落差

比較決策樹 max_depth=3、None，以及100棵樹的隨機森林（min_samples_leaf=2）。全部 random_state=42。
使用同一套前處理與同一批資料，列出 train_MAE、valid_MAE；以驗證 MAE 選一個。
**驗收：** 不以訓練 MAE 最低作為唯一理由，不使用測試集選模型。


In [6]:
# 把候選模型放入字典，記錄分數與已 fit 的完整流程。
candidates = {
    "Tree depth 3": DecisionTreeRegressor(max_depth=3, random_state=42),
    "Tree unlimited": DecisionTreeRegressor(random_state=42),
    "Forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1),
}
trained = {}
rows = []
for name, regressor in candidates.items():
    estimator = Pipeline([("prep", make_preprocessor()), ("reg", regressor)]).fit(X_train, y_train)
    trained[name] = estimator
    rows.append({"name": name,
                 "train_MAE": mean_absolute_error(y_train, estimator.predict(X_train)),
                 "valid_MAE": mean_absolute_error(y_valid, estimator.predict(X_valid))})
scores = pd.DataFrame(rows).set_index("name").sort_values("valid_MAE")
display(scores.round(2))
winner = scores.index[0]
print("依驗證 MAE 選擇:", winner)


,train_MAE,valid_MAE
name,,
Forest,1120.34,2419.02
Tree depth 3,2880.08,3704.01
Tree unlimited,0.00,3928.88


依驗證 MAE 選擇: Forest


## 作業 6｜最終評估與新資料

clone 作業5選定模型，在 train+valid 合併的開發集重新 fit。只此一次用測試集計算四個指標。
預測一間28坪、屋齡缺失、捷運500公尺、2房、6樓、East、Yes電梯的房屋。列出預測單位並重啟Kernel全部執行。
**驗收：** 新房屋有完整欄名；未手動補值；未重新fit新房屋；測試評估後不再調參。


In [7]:
# clone 只帶設定，在開發資料重學規則，最後測試只評估。
from sklearn.base import clone
final_model = clone(trained[winner]).fit(X_dev, y_dev)
final_pred = final_model.predict(X_test)
display(pd.Series({
    "MAE": mean_absolute_error(y_test, final_pred),
    "MSE": mean_squared_error(y_test, final_pred),
    "RMSE": root_mean_squared_error(y_test, final_pred),
    "R2": r2_score(y_test, final_pred),
}))
new_house = pd.DataFrame([{
    "area_ping": 28, "building_age": np.nan, "distance_mrt_m": 500,
    "bedrooms": 2, "floor": 6, "district": "East", "has_elevator": "Yes",
}])[feature_columns]
print("新房屋預測月租金（元）:", final_model.predict(new_house)[0])


MAE     1.846484e+03
MSE     5.321022e+06
RMSE    2.306734e+03
R2      9.607482e-01
dtype: float64

新房屋預測月租金（元）: 28419.193035714285
